# MESACLIP High-Resolution Ocean: Regridding Sea Surface Temperature

```{image} ../../thumbnails/gdex_logo.png
:alt: GDEX Cookbook logo
:width: 200px
```

---

## Overview

MESACLIP is a high-resolution iHESP CESM climate simulation whose ocean component
runs on the 0.1° POP **curvilinear** grid. Comparing or mapping that output usually
means regridding it to a regular latitude–longitude grid first. GDEX hosts the run
as an ARCO dataset [`d651007`](https://gdex.ucar.edu/datasets/d651007/).

Here we open daily sea surface temperature from a kerchunk reference, regrid one
snapshot from the native POP grid to 1°×1° using precomputed ESMF weights, and
display it on an interactive map.

1. Open MESACLIP ocean SST from a kerchunk reference over OSDF
2. Select a single SST time slice from the ~67 TB archive
3. Regrid from the POP curvilinear grid to 1°×1° with ESMF weights
4. Visualize the result on an interactive Folium map

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [ERA5 Reanalysis Data Workflow](era5.ipynb) | Helpful | The kerchunk reference access pattern |
| [xarray](https://docs.xarray.dev) | Necessary | Labeled N-D arrays |
| Regridding / ESMF weights | Helpful | Bilinear remapping concepts |
| [rioxarray](https://corteva.github.io/rioxarray/) + [folium](https://python-visualization.github.io/folium/) | Necessary | Reprojection and interactive mapping |
| [SciPy sparse](https://docs.scipy.org/doc/scipy/reference/sparse.html) | Necessary | Applying the regridding weights |

- **Time to learn**: 40 minutes
- **Level**: advanced

---

## Imports

In [1]:
import fsspec
import numpy as np
import scipy.sparse as sps
import xarray as xr
import matplotlib.pyplot as plt
import rioxarray  # noqa: F401 — activates the .rio accessor
import folium
import branca.colormap as cm
from netCDF4 import default_fillvals

## Locating the dataset

The full MESACLIP ocean output is spread across many NetCDF files. GDEX provides a
single kerchunk reference that indexes all variables and time steps for one
ensemble member, so we only open one file to reach the whole archive.

In [6]:
reference_url = (
    "https://data.gdex.ucar.edu/d651007/kerchunk/"
    "b.e13.BHISTC5.ne120_t12.cesm-ihesp-hires1.0.46-1920-2005.010.ocn.day_1-remote-osdf.parq"
)
reference_url

'https://data.gdex.ucar.edu/d651007/kerchunk/b.e13.BHISTC5.ne120_t12.cesm-ihesp-hires1.0.46-1920-2005.010.ocn.day_1-remote-osdf.parq'

### Opening the data

Opening the reference is cheap — it loads a lookup table of chunk locations, not
the data itself. `chunks={}` keeps everything lazy, so the array represents the
full multi-terabyte archive while almost nothing is in memory yet.

In [7]:
%%time
ds = xr.open_dataset(reference_url, engine="kerchunk",decode_timedelta=False,chunks={},
)
ds

CPU times: user 3.63 s, sys: 1.74 s, total: 5.37 s
Wall time: 8.03 s


<xarray.Dataset> Size: 74TB
Dimensions:             (time: 31390, nlat: 2400, nlon: 3600, z_t: 62, z_w: 62,
                         d2: 2, z_t_150m: 15, z_w_bot: 62, z_w_top: 62)
Coordinates:
  * time                (time) object 251kB 1920-01-02 00:00:00 ... 2006-01-0...
    TLAT                (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    TLONG               (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    ULAT                (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    ULONG               (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
  * z_t                 (z_t) float32 248B 500.0 1.5e+03 ... 5.625e+05 5.875e+05
  * z_w                 (z_w) float32 248B 0.0 1e+03 2e+03 ... 5.5e+05 5.75e+05
  * z_t_150m            (z_t_150m) float32 60B 500.0 1.5e+03 ... 1.45e+04
  * z_w_bot             (z_w_bot) float32 248B 1e+03 2e+03 ... 5.75e+05 6e+05
  * z_w_top             (z_w_top) float32 248B 0.0 1e+03 ... 5.5e+05 5.75e+05
Dimensions without coordinates: nlat, nlon, d2
Data variables: (12/76)
    ANGLE               (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    ANGLET              (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    DXT                 (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    DXU                 (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    DYT                 (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    DYU                 (time, nlat, nlon) float64 2TB dask.array<chunksize=(1, 480, 720), meta=np.ndarray>
    ...                  ...
    sea_ice_salinity    (time) float64 251kB dask.array<chunksize=(1,), meta=np.ndarray>
    sflux_factor        (time) float64 251kB dask.array<chunksize=(1,), meta=np.ndarray>
    sound               (time) float64 251kB dask.array<chunksize=(1,), meta=np.ndarray>
    stefan_boltzmann    (time) float64 251kB dask.array<chunksize=(1,), meta=np.ndarray>
    time_bound          (time, d2) object 502kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    vonkar              (time) float64 251kB dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    title:           b.e13.BHISTC5.ne120_t12.cesm-ihesp-hires1.0.46-1920-2005...
    history:         none
    Conventions:     CF-1.0; http://www.cgd.ucar.edu/cms/eaton/netcdf/CF-curr...
    contents:        Diagnostic and Prognostic Variables
    source:          CCSM POP2, the CCSM Ocean Component
    revision:        $Id: tavg.F90 56176 2013-12-20 18:35:46Z mlevy@ucar.edu $
    calendar:        All years have exactly  365 days.
    start_time:      This dataset was created on 2023-11-30 at 21:10:48.7
    cell_methods:    cell_methods = time: mean ==> the variable values are av...
    nsteps_total:    611
    tavg_sum:        84600.0
    tavg_sum_qflux:  84600.0

In [8]:
print(f"{ds.nbytes / 1024**4:.1f} TB referenced by a single file")

67.1 TB referenced by a single file


## Selecting a single SST time slice

We pull one daily SST field into memory, along with the grid's 2D `TLAT`/`TLONG`
coordinates (constant in time). The POP grid stores land as large negative values,
which we mask out.

In [9]:
%%time
da_sst = ds["SST"].isel(time=1030).load()
da_lat = ds["TLAT"].isel(time=0).load()
da_lon = ds["TLONG"].isel(time=0).load()

ds_sst = xr.Dataset()
ds_sst["SST"] = da_sst
ds_sst["TLONG"] = da_lon
ds_sst["TLAT"] = da_lat
ds_sst = ds_sst.expand_dims("time")

ds_sst["SST"] = ds_sst["SST"].where(ds_sst["SST"].compute() > -1)
ds_sst

TypeError: argument of type 'NAType' is not iterable

In [ ]:
ds_sst["SST"].isel(time=0).plot()

## Regridding from the POP grid

We regrid with precomputed **ESMF bilinear weights** (prepared by NCAR's scientist Frederic
Castruccio) that map the POP tx0.1v2 grid onto a regular 1°×1° grid. The weights
are applied as a single sparse matrix multiplication, then normalized by fractional
ocean coverage to remove land contamination near coastlines.